# Solar Flare ML Prediction - Data Overview and Exploration

This notebook provides an overview of the solar flare prediction dataset and explores the McIntosh classification features.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Data

The project uses McIntosh classification data for two solar cycles:
- **Solar Cycle 22** (Training data)
- **Solar Cycle 23** (Test data)

In [ ]:
# Load datasets
names = ['mcint', 'mcint_evol', 'class']
df_train = pd.read_csv('../mcint_ml22.csv', names=names, dtype={'mcint': str, 'mcint_evol': str, 'class': np.float64})
df_test = pd.read_csv('../mcint_ml23.csv', names=names, dtype={'mcint': str, 'mcint_evol': str, 'class': np.float64})

print(f"Training set shape (Solar Cycle 22): {df_train.shape}")
print(f"Test set shape (Solar Cycle 23): {df_test.shape}")
print("\nFirst 10 rows of training data:")
df_train.head(10)

## 3. Class Distribution

Analyze the target variable (flare occurrence within 24 hours)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set class distribution
class_counts_train = df_train['class'].value_counts().sort_index()
axes[0].bar(['No Flare (0)', 'Flare (1)'], class_counts_train.values, color=['steelblue', 'coral'])
axes[0].set_title('Solar Cycle 22 - Class Distribution (Training)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Class')
for i, v in enumerate(class_counts_train.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Test set class distribution
class_counts_test = df_test['class'].value_counts().sort_index()
axes[1].bar(['No Flare (0)', 'Flare (1)'], class_counts_test.values, color=['steelblue', 'coral'])
axes[1].set_title('Solar Cycle 23 - Class Distribution (Test)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Class')
for i, v in enumerate(class_counts_test.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nTraining Set (Cycle 22):")
print(f"  No Flare: {class_counts_train[0.0]} ({class_counts_train[0.0]/len(df_train)*100:.2f}%)")
print(f"  Flare: {class_counts_train[1.0]} ({class_counts_train[1.0]/len(df_train)*100:.2f}%)")

print(f"\nTest Set (Cycle 23):")
print(f"  No Flare: {class_counts_test[0.0]} ({class_counts_test[0.0]/len(df_test)*100:.2f}%)")
print(f"  Flare: {class_counts_test[1.0]} ({class_counts_test[1.0]/len(df_test)*100:.2f}%)")

## 4. McIntosh Classification System

The McIntosh classification has two components:
- **Static classification (mcint)**: Initial sunspot configuration (3 digits)
- **Evolution codes (mcint_evol)**: How the sunspot changed (6 digits)

In [ ]:
# Analyze McIntosh classifications
print("Unique McIntosh Static Classifications (Training Set):")
unique_mcint = df_train['mcint'].nunique()
print(f"  Total unique: {unique_mcint}")
print(f"\nTop 10 most common:")
print(df_train['mcint'].value_counts().head(10))

print(f"\n\nUnique Evolution Codes (Training Set):")
unique_evol = df_train['mcint_evol'].nunique()
print(f"  Total unique: {unique_evol}")
print(f"\nTop 10 most common:")
print(df_train['mcint_evol'].value_counts().head(10))

## 5. Feature Engineering - Parse Evolution Codes

Evolution codes are 6 characters representing:
- Characters 0-2: Starting Zurich-Penetration-Class (ZPC)
- Characters 3-5: Ending ZPC
- Each ZPC has 3 components: Z (Zurich), P (Penetration), C (Compactness)

In [ ]:
# Parse evolution codes into components
df_train_features = df_train.copy()
df_train_features['z1'] = df_train_features['mcint_evol'].str[0]
df_train_features['p1'] = df_train_features['mcint_evol'].str[1]
df_train_features['c1'] = df_train_features['mcint_evol'].str[2]
df_train_features['z2'] = df_train_features['mcint_evol'].str[3]
df_train_features['p2'] = df_train_features['mcint_evol'].str[4]
df_train_features['c2'] = df_train_features['mcint_evol'].str[5]

print("Parsed Evolution Code Components (first 10 rows):")
print(df_train_features[['mcint_evol', 'z1', 'p1', 'c1', 'z2', 'p2', 'c2', 'class']].head(10))

## 6. Feature Component Analysis

Analyze the distribution of individual components in evolution codes

In [ ]:
# Analyze feature components
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Evolution Code Component Distribution (Starting Configuration)', fontsize=14, fontweight='bold')

components = ['z1', 'p1', 'c1', 'z2', 'p2', 'c2']
titles = ['Zurich (Start)', 'Penetration (Start)', 'Compactness (Start)',
          'Zurich (End)', 'Penetration (End)', 'Compactness (End)']

for idx, (ax, comp, title) in enumerate(zip(axes.flat, components, titles)):
    value_counts = df_train_features[comp].value_counts().sort_index()
    ax.bar(range(len(value_counts)), value_counts.values, color='skyblue', edgecolor='navy')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.set_xticks(range(len(value_counts)))
    ax.set_xticklabels(value_counts.index, rotation=45)

plt.tight_layout()
plt.show()

## 7. Feature-Target Relationship

Analyze how features correlate with flare occurrence

In [ ]:
# Analyze flare occurrence for each component
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Flare Occurrence by Component', fontsize=14, fontweight='bold')

for idx, (ax, comp) in enumerate(zip(axes.flat, components)):
    flare_rate = df_train_features.groupby(comp)['class'].agg(['sum', 'count'])
    flare_rate['rate'] = (flare_rate['sum'] / flare_rate['count'] * 100).round(2)
    
    colors = ['coral' if x > 15 else 'steelblue' for x in flare_rate['rate']]
    ax.bar(range(len(flare_rate)), flare_rate['rate'].values, color=colors, edgecolor='black')
    ax.set_title(titles[idx], fontweight='bold')
    ax.set_xlabel('Component Value')
    ax.set_ylabel('Flare Rate (%)')
    ax.set_xticks(range(len(flare_rate)))
    ax.set_xticklabels(flare_rate.index, rotation=45)
    ax.axhline(df_train_features['class'].mean() * 100, color='red', linestyle='--', alpha=0.5, label='Overall Rate')

plt.tight_layout()
plt.show()

print(f"Overall flare rate in training data: {df_train_features['class'].mean()*100:.2f}%")

## 8. Data Statistics Summary

In [ ]:
print("="*60)
print("DATASET SUMMARY")
print("="*60)

print(f"\nTrain Set (Solar Cycle 22):")
print(f"  Samples: {len(df_train)}")
print(f"  Flare events: {df_train['class'].sum():.0f}")
print(f"  Flare rate: {df_train['class'].mean()*100:.2f}%")

print(f"\nTest Set (Solar Cycle 23):")
print(f"  Samples: {len(df_test)}")
print(f"  Flare events: {df_test['class'].sum():.0f}")
print(f"  Flare rate: {df_test['class'].mean()*100:.2f}%")

print(f"\nFeatures Used:")
print(f"  McIntosh static classification (1)")
print(f"  McIntosh evolution code (1)")
print(f"  Parsed components (6): Z1, P1, C1, Z2, P2, C2")
print(f"  Encoding options: standard, one-hot encoded, or combinations")

print(f"\nTarget Variable:")
print(f"  0 = No X-ray flare within 24 hours")
print(f"  1 = X-ray flare occurred within 24 hours")
print("="*60)